In [73]:
import pandas as pd
import librosa 
from pathlib import Path
import numpy as np 
import soundfile as sf
from IPython.display import Audio
import scipy.signal as sig
from tqdm import tqdm 
import h5py

import sys
# get audio utils
sys.path.append('/om2/user/imgriff/datasets/spatial_audio_pipeline/utils')
import util_audio

# Make stimuli for CommonVoice word recognition experimeent

Word recognition stimuli for human experiment run on prolific.

We will use the foreground signals defined in the manifest:    
`/om2/user/imgriff/datasets/commonvoice_9/en/manifeests/cv_model_and_participant_eval_manifest.pdpkl`

Which has signals generated and stored in the h5 here:   
`/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/model_and_participant_test_set/model_and_participant_test_set_50000Hz_rate_60dB_000.hdf5`


For a first pass, the experiment will only have 3 conditions:
* Clean speech
* Speech-shaped noise at 0dB
* Audioset Scenes at 0dB

In [83]:
## Define windowing function - will apply a cosine ramp to the start and end of a signal

def ramp_hann(x, ramp_dur_ms, samplerate=20_000):
    stim_dur_smp = x.shape[0] # N taps of x
    ramp_dur_smp =  np.floor(ramp_dur_ms * samplerate / 1000).astype('int')
    assert stim_dur_smp > (2*ramp_dur_smp), 'Ramps cannot be longer than the stimulus duration'
    
    # calc window
    # https://stackoverflow.com/questions/56485663/hanning-window-values-doesnt-match-in-python-and-matlab
    win = sig.hann((2 * ramp_dur_smp) + 2)[1:-1]
    
    # Beginning of windowed stimulus
    start_win = win[ : ramp_dur_smp ] * x[ : ramp_dur_smp]
    
    # Middle part (steady state)
    steady_win = x[ramp_dur_smp : stim_dur_smp-ramp_dur_smp]
    
    # Final part of windowed stimulus
    end_win = win[ramp_dur_smp : ramp_dur_smp*2] * x[stim_dur_smp-ramp_dur_smp:stim_dur_smp]

    return np.hstack([start_win, steady_win, end_win])

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))


def generate_speech_shaped_noise(signal):
    # get spectrum
    n_fft=next_pow_2(len(signal))
    X = np.fft.rfft(signal, n=n_fft)
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    ssn = noise[:len(signal)]
    return ssn

### Get AudioSet backgrounds 

In [4]:
# path_to_jsin_eval_mfest = '/om4/group/mcdermott/user/jfeather/audioset_dataframes/srNative/cullLabels_silence/eval_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdh5'

# jsin_meta_df = pd.read_hdf(path_to_jsin_eval_mfest)

In [5]:
# jsin_meta_df['path'] = jsin_meta_df['path'].str.replace('all', '0')

In [6]:
# jsin_meta_df

In [7]:
# new_out_path = '/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/audioset/jsin_eval_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdpkl'
# jsin_meta_df.to_pickle(new_out_path)

In [9]:
path_to_jsin_eval_mfest = '/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/audioset/jsin_eval_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdpkl'

jsin_meta_df = pd.read_pickle(path_to_jsin_eval_mfest)

In [23]:
### Get audioset ontology map

"""
Load audioset ontology and build dictionaries to map:

audioset_map_mid_to_label: ID to label
audioset_map_label_to_mid: label to ID
audioset_map_mid_to_parent: IDs to one parent ID
audioset_map_mid_to_children: IDs to list of child IDs
"""
import os 
import json

def dfs(visited, graph, node):
    """
    Depth-first search
    """
    if node not in visited:
        visited.add(node)
        for neighbour in graph.get(node, []):
            dfs(visited, graph, neighbour)

            
dir_audioset = '/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/audioset/'
with open(os.path.join(dir_audioset, 'ontology.json'), 'r') as f:
    ontology = json.load(f)

audioset_map_mid_to_label = {}
audioset_map_label_to_mid = {}
audioset_map_mid_to_parent = {}
audioset_map_mid_to_children = {}
for d0 in ontology:
    audioset_map_mid_to_label[d0['id']] = d0['name']
    audioset_map_label_to_mid[d0['name']] = d0['id']
    for d1 in ontology:
        if d0['id'] in d1['child_ids']:
            if d0['id'] not in audioset_map_mid_to_parent:
                audioset_map_mid_to_parent[d0['id']] = set([])
            audioset_map_mid_to_parent[d0['id']].add(d1['id'])
            if d1['id'] not in audioset_map_mid_to_children:
                audioset_map_mid_to_children[d1['id']] = set([])
            audioset_map_mid_to_children[d1['id']].add(d0['id'])
for k in sorted(audioset_map_mid_to_parent.keys()):
    if len(audioset_map_mid_to_parent[k]) == 1:
        audioset_map_mid_to_parent[k] = list(audioset_map_mid_to_parent[k])[0]
    else:
        audioset_map_mid_to_parent.pop(k)

INVALID_SEGMENT_ROOTS = [
    'Speech',
    'Singing',
    'Vocal music',
    'Whispering',
    'Shout',
    'Music'
]
INVALID_SEGMENT_LABELS = []

for label in INVALID_SEGMENT_ROOTS:
    INVALID_SEGMENT_LABELS.append(label)
    mid = audioset_map_label_to_mid[label]
    list_child_mid = set()
    dfs(list_child_mid, audioset_map_mid_to_children, mid)
    list_child_mid = list(list_child_mid)
    # print(audioset_map_mid_to_label[mid])
    for child_mid in list_child_mid:
        # print('|__', audioset_map_mid_to_label[child_mid])
        INVALID_SEGMENT_LABELS.append(audioset_map_mid_to_label[child_mid])
    
INVALID_SEGMENT_LABELS = list(set(INVALID_SEGMENT_LABELS))

INVALID_SEGMENT_LABELS = [label.lower() for label in INVALID_SEGMENT_LABELS]



In [28]:
invalid_labels = '|'.join(INVALID_SEGMENT_LABELS)

In [99]:
scene_manifest = jsin_meta_df[~jsin_meta_df.labels_decoded_slugged.str.contains(invalid_labels)].reset_index()
music_manifest = jsin_meta_df[jsin_meta_df.has_music_label == True]

/tmp/ipykernel_523219/2219300421.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  scene_manifest = jsin_meta_df[~jsin_meta_df.labels_decoded_slugged.str.contains(invalid_labels)].reset_index()


In [96]:
scene_manifest.shape

(8614, 25)

In [35]:
# Get natural scene example
wav, sr = librosa.load(scene_manifest.iloc[0].path.split('?')[0], sr=None)
Audio(wav, rate=sr)

In [36]:
# Get music example 

# Get natural scene example
wav, sr = librosa.load(music_manifest.iloc[0].path.split('?')[0], sr=None)
Audio(wav, rate=sr)

# Get Commonvoice foregrounds

In [45]:
import pickle
## Import cv vocab and test manifest
commonvoice_path = Path('/om2/user/imgriff/datasets/commonvoice_9/en/')


vocab_fn = commonvoice_path / 'cv_word_int_label_dict.pkl'

with open(vocab_fn, 'rb') as handle:
    cv_vocab = pickle.load(handle)

cv_class_map = {v:k for k,v in cv_vocab.items()}

In [38]:
h5_path = "/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/model_and_participant_test_set/model_and_participant_test_set_50000Hz_rate_60dB_000.hdf5"

cv_h5 = h5py.File(h5_path, 'r')

In [47]:
print(cv_class_map[cv_h5['label_word_int'][0]])
Audio(cv_h5['signal'][0], rate=50000)

heavy


## Make trial conditions 

Each foreground will be generated in each condition.

The output file structure will be:   
`../{condition_name}/{word_class_ix}_{eg_ix}.wav`



In [109]:
import itertools

# setup design
snrs = [0]

noise_conditions = ['clean', 'stationary_noise', 'natural_scene']

condition_pairs = list(itertools.product(snrs, noise_conditions))

condition_dict = {f"condition{ix:02}":pair for ix, pair in enumerate(condition_pairs)}
condition_dict


exp_dir = Path('/om2/user/imgriff/projects/Auditory-Attention/behavioral_experiments/commonvoice_word_rec/')
if not exp_dir.is_dir():
    exp_dir.mkdir(exist_ok=True, parents=True)

print(condition_dict)
with open(exp_dir/'pilot_condiition_dict.pkl', 'wb') as handle:
    pickle.dump(condition_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


{'condition00': (0, 'clean'), 'condition01': (0, 'stationary_noise'), 'condition02': (0, 'natural_scene')}


In [60]:
def center_crop(sig, sr, out_dur=2):
    assert len(np.array(sig).shape) == 1, "Only works on 1d array"

    n_dur_frames = int(out_dur * sr)
    dur_diff = np.abs(len(sig) - n_dur_frames)
    return sig[int(np.floor(dur_diff / 2)) : -int(np.ceil(dur_diff / 2))]


In [65]:
print(cv_class_map[cv_h5['label_word_int'][0]])
Audio(cv_h5['signal'][0], rate=50000)

heavy


In [67]:
# Get natural scene example
wav_eg = center_crop(cv_h5['signal'][0], 50_000)
Audio(wav_eg, rate=50_000)


In [106]:
scene_manifest = scene_manifest[scene_manifest.has_music_label == False]

In [183]:
np.random.seed(0)

path_to_write = Path('/mindhive/mcdermott/www/imgriff/msjspsych/commonvoice_word_recognition/stim/v01_cv_eval/')

n_foregrounds = len(cv_h5['signal'])

dur_in_s = 2 # 2 second duration
dBSPL = 60 
snr_dB = 0 

orig_samp_rate = 50_000
out_samp_rate = 20_000 

audio_manifest = []
def center_crop(sig, sr, out_dur=2):
    assert len(np.array(sig).shape) == 1, "Only works on 1d array"
    n_dur_frames = int(out_dur * sr)
    dur_diff = np.abs(len(sig) - n_dur_frames)
    return sig[int(np.floor(dur_diff / 2)) : -int(np.ceil(dur_diff / 2))]


for n in range(n_foregrounds):
# n=10
# get foreground and labels for ix n 
    foreground = cv_h5['signal'][n]
    # downsample to new rate
    foreground = librosa.resample(foreground, orig_sr=orig_samp_rate, target_sr=out_samp_rate)
    #convert to 2 seconds
    foreground = center_crop(foreground, out_samp_rate)

    # get labels 
    word_int = cv_h5['label_word_int'][n]
    word = cv_class_map[word_int]

    # get clean signal
    clean_audio = ramp_hann(foreground, ramp_dur_ms=10, samplerate=out_samp_rate)
    clean_audio = util_audio.set_dBSPL(clean_audio, dBSPL=dBSPL, mean_subtract=True)
    # write to stim dir
    out_name = path_to_write / f'condition00/{word_int:03}_000.wav'
    if not out_name.parent.exists():
        out_name.parent.mkdir(exist_ok=True, parents=True)
    sf.write(out_name.as_posix(), clean_audio, out_samp_rate, subtype='PCM_24') 

    # get speech-shaped noise example 
    ssn = generate_speech_shaped_noise(foreground)
    sig_w_ssn, level_scale_factor = combine_with_noise(foreground, ssn, snr=snr_dB) 
    sig_w_ssn = ramp_hann(sig_w_ssn, ramp_dur_ms=10, samplerate=out_samp_rate)
    sig_w_ssn = util_audio.set_dBSPL(sig_w_ssn, dBSPL=dBSPL, mean_subtract=True)
    # write
    out_name = path_to_write / f'condition01/{word_int:03}_000.wav'
    if not out_name.parent.exists():
        out_name.parent.mkdir(exist_ok=True, parents=True)
    sf.write(out_name.as_posix(), sig_w_ssn, out_samp_rate, subtype='PCM_24') 


    # get audioset example 
    audioset_eg = scene_manifest.sample(1)
    aud_start_time = np.round(np.random.uniform(0, audioset_eg.end_secs.item() - 2), 2 ) 
    aud_path_name, meta = audioset_eg.path.item().split('?')
    audioset_eg_audio, _ = librosa.load(aud_path_name, sr=out_samp_rate, offset=aud_start_time, duration=dur_in_s)
    # combine with foreground
    sig_w_audset, _ = combine_with_noise(foreground, audioset_eg_audio, snr=snr_dB) 
    sig_w_audset = ramp_hann(sig_w_audset, ramp_dur_ms=10, samplerate=out_samp_rate)
    sig_w_audset = util_audio.set_dBSPL(sig_w_audset, dBSPL=dBSPL, mean_subtract=True)
    # write 
    out_name = path_to_write / f'condition02/{word_int:03}_000.wav'
    if not out_name.parent.exists():
        out_name.parent.mkdir(exist_ok=True, parents=True)
    sf.write(out_name.as_posix(), sig_w_audset, out_samp_rate, subtype='PCM_24') 

    audio_manifest.append({
                'wav_dir': path_to_write,
                'parent_h5': h5_path,
                'h5_ix' : n,
                'signal_dur_in_s': dur_in_s,
                'sr': out_samp_rate,
                'word_int': word_int,
                'word': word,
                'clean_cond': 'condition00',
                'stationary_noise_cond': 'condition01',
                'audioset_cond': 'condition02',
                'audioset_manifest':  path_to_jsin_eval_mfest,
                'audioset_manifest_ix': audioset_eg['index'].item(),
                'audioset_eg_start_in_s': aud_start_time
    })



In [184]:
audio_manifest_df = pd.DataFrame.from_records(audio_manifest)

In [185]:
audio_manifest_df.to_pickle(exp_dir/'pilot_stimuli_to_condition_manifest.pdpkl')

In [186]:
!ls {exp_dir}

pilot_condiition_dict.pkl  pilot_stimuli_to_condition_manifest.pdpkl


In [139]:
print(word)
Audio(sig_w_audset, rate=out_samp_rate, normalize=False)

recently


In [193]:
Audio('/mindhive/mcdermott/www/imgriff/msjspsych/commonvoice_word_recognition/stim/v01_cv_eval/condition01/157_000.wav')